In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

np.random.seed(42)
print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [2]:
# What each platform CLAIMS to pay (from their actual partner pages)
platforms = {
    'Swiggy':        {'claimed': 40000, 'base_per_trip': 35,  'trips_per_day': 18, 'incentive_rate': 0.22},
    'Zomato':        {'claimed': 35000, 'base_per_trip': 33,  'trips_per_day': 17, 'incentive_rate': 0.20},
    'Rapido':        {'claimed': 50000, 'base_per_trip': 28,  'trips_per_day': 22, 'incentive_rate': 0.28},
    'Urban Company': {'claimed': 45000, 'base_per_trip': 280, 'trips_per_day': 4,  'incentive_rate': 0.15},
    'Porter':        {'claimed': 38000, 'base_per_trip': 95,  'trips_per_day': 8,  'incentive_rate': 0.18},
}

# City multipliers based on real cost of living differences
cities = {
    'Hyderabad': {'multiplier': 1.00, 'fuel_cost': 130},
    'Mumbai':    {'multiplier': 1.18, 'fuel_cost': 145},
    'Bengaluru': {'multiplier': 1.12, 'fuel_cost': 138},
    'Delhi':     {'multiplier': 1.08, 'fuel_cost': 142},
    'Chennai':   {'multiplier': 0.92, 'fuel_cost': 128},
}

print("✅ Platform and city configs loaded")
print(f"\nPlatforms: {list(platforms.keys())}")
print(f"Cities: {list(cities.keys())}")

✅ Platform and city configs loaded

Platforms: ['Swiggy', 'Zomato', 'Rapido', 'Urban Company', 'Porter']
Cities: ['Hyderabad', 'Mumbai', 'Bengaluru', 'Delhi', 'Chennai']


In [3]:
def is_peak(hour):
    """Peak hours: morning 7-10am, lunch 12-2pm, evening 6-10pm"""
    return (7 <= hour <= 10) or (12 <= hour <= 14) or (18 <= hour <= 22)

def peak_factor(hour):
    """Earnings multiplier based on time of day"""
    return 1.35 if is_peak(hour) else 0.72

def get_experience_multiplier(exp):
    """Experienced workers earn more per trip"""
    return {'< 6 months': 0.78, '6-12 months': 0.92,
            '1-2 years': 1.05, '2+ years': 1.15}[exp]

print("✅ Helper functions defined")
print("\nPeak hours: 7-10am, 12-2pm, 6-10pm")
print("Off-peak multiplier: 0.72x")
print("Peak multiplier:     1.35x")

✅ Helper functions defined

Peak hours: 7-10am, 12-2pm, 6-10pm
Off-peak multiplier: 0.72x
Peak multiplier:     1.35x


In [4]:
records = []
worker_id = 1

for platform, pdata in platforms.items():
    print(f"Generating data for {platform}...")
    
    for w in range(400):  # 400 workers per platform = 2000 total
        
        # Assign city (Hyderabad weighted higher since we're based there)
        city = np.random.choice(
            list(cities.keys()),
            p=[0.28, 0.22, 0.22, 0.18, 0.10]
        )
        cdata = cities[city]
        
        experience = np.random.choice(
            ['< 6 months', '6-12 months', '1-2 years', '2+ years'],
            p=[0.25, 0.30, 0.28, 0.17]
        )
        
        vehicle = np.random.choice(
            ['Bike', 'Bicycle', 'Car'],
            p=[0.72, 0.08, 0.20]
        ) if platform != 'Urban Company' else np.random.choice(
            ['Bike', 'Car'], p=[0.4, 0.6]
        )
        
        exp_mult = get_experience_multiplier(experience)
        
        # Simulate 26 working days per month
        for day in range(26):
            date = datetime(2024, 3, 1) + timedelta(days=day)
            
            # Random shift: 8 to 12 hours
            shift_start = np.random.choice([6,7,8,9,10], p=[0.10,0.25,0.35,0.20,0.10])
            shift_len   = np.random.randint(8, 13)
            work_hours  = list(range(shift_start, min(shift_start + shift_len, 24)))
            
            daily_earnings   = 0
            daily_trips      = 0
            daily_idle_time  = 0
            daily_incentives = 0
            
            for h in work_hours:
                pf = peak_factor(h)
                
                # Trips this hour
                base_trips = pdata['trips_per_day'] / 10
                trips_this_hour = max(0, np.random.poisson(base_trips * pf * exp_mult))
                
                # Earnings per trip with realistic noise
                ept = pdata['base_per_trip'] * cdata['multiplier']
                ept = ept * (1 + np.random.normal(0, 0.12))
                ept = max(ept * 0.6, ept)
                
                # Incentives (not always paid — designed to be hard to achieve)
                incentive = 0
                if trips_this_hour >= 2:
                    incentive = (ept * pdata['incentive_rate'] * 
                                trips_this_hour * 
                                np.random.choice([0,0,0,1,1,1], p=[0.15,0.15,0.10,0.25,0.25,0.10]))
                
                # Idle time in minutes (waiting between orders)
                idle = max(0, 60 - trips_this_hour * 12 - np.random.randint(0, 15))
                idle = min(idle, 55)
                
                daily_earnings   += trips_this_hour * ept + incentive
                daily_trips      += trips_this_hour
                daily_idle_time  += idle
                daily_incentives += incentive
            
            # Daily costs every worker pays out of pocket
            fuel_daily  = cdata['fuel_cost'] if vehicle == 'Bike' else cdata['fuel_cost'] * 1.8 if vehicle == 'Car' else 0
            phone_daily = 15
            depreciation= 35 if vehicle == 'Bike' else 80 if vehicle == 'Car' else 5
            total_costs = fuel_daily + phone_daily + depreciation
            
            net_earnings     = max(0, daily_earnings - total_costs)
            hours_worked     = len(work_hours)
            effective_hourly = net_earnings / hours_worked if hours_worked > 0 else 0
            gross_hourly     = daily_earnings / hours_worked if hours_worked > 0 else 0
            
            records.append({
                'worker_id':            f'W{worker_id:05d}',
                'platform':             platform,
                'city':                 city,
                'date':                 date.strftime('%Y-%m-%d'),
                'day_of_week':          date.strftime('%A'),
                'experience':           experience,
                'vehicle':              vehicle,
                'hours_worked':         hours_worked,
                'trips_completed':      daily_trips,
                'gross_earnings':       round(daily_earnings, 2),
                'incentives_earned':    round(daily_incentives, 2),
                'fuel_cost':            round(fuel_daily, 2),
                'phone_cost':           round(phone_daily, 2),
                'depreciation':         round(depreciation, 2),
                'total_costs':          round(total_costs, 2),
                'net_earnings':         round(net_earnings, 2),
                'idle_time_mins':       round(daily_idle_time, 1),
                'earnings_per_trip':    round(daily_earnings / max(daily_trips, 1), 2),
                'effective_hourly_rate':round(effective_hourly, 2),
                'gross_hourly_rate':    round(gross_hourly, 2),
                'is_peak_day':          1 if date.weekday() in [4, 5] else 0,
            })
        
        worker_id += 1

df = pd.DataFrame(records)
print(f"\n✅ Dataset generated!")
print(f"Total rows: {len(df):,}")
print(f"Workers:    {df['worker_id'].nunique():,}")
print(f"Platforms:  {df['platform'].nunique()}")
print(f"Cities:     {df['city'].nunique()}")

Generating data for Swiggy...
Generating data for Zomato...
Generating data for Rapido...
Generating data for Urban Company...
Generating data for Porter...

✅ Dataset generated!
Total rows: 52,000
Workers:    2,000
Platforms:  5
Cities:     5


In [5]:
# Create processed folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

# Save daily data
df.to_csv('../data/processed/gig_daily.csv', index=False)
print(f"✅ Saved: data/processed/gig_daily.csv ({len(df):,} rows)")

# Monthly summary per worker
monthly = df.groupby(['worker_id','platform','city','experience','vehicle']).agg(
    monthly_gross=('gross_earnings','sum'),
    monthly_net=('net_earnings','sum'),
    monthly_incentives=('incentives_earned','sum'),
    monthly_hours=('hours_worked','sum'),
    monthly_trips=('trips_completed','sum'),
    avg_effective_hourly=('effective_hourly_rate','mean'),
    avg_efficiency=('earnings_per_trip','mean'),
    avg_idle_mins=('idle_time_mins','mean'),
).reset_index().round(2)

monthly.to_csv('../data/processed/gig_monthly.csv', index=False)
print(f"✅ Saved: data/processed/gig_monthly.csv ({len(monthly):,} rows)")

# Platform claims file
claims = pd.DataFrame([
    {'platform': 'Swiggy',        'claimed_monthly': 40000},
    {'platform': 'Zomato',        'claimed_monthly': 35000},
    {'platform': 'Rapido',        'claimed_monthly': 50000},
    {'platform': 'Urban Company', 'claimed_monthly': 45000},
    {'platform': 'Porter',        'claimed_monthly': 38000},
])
claims.to_csv('../data/raw/platform_claims.csv', index=False)
print(f"✅ Saved: data/raw/platform_claims.csv")

✅ Saved: data/processed/gig_daily.csv (52,000 rows)
✅ Saved: data/processed/gig_monthly.csv (2,000 rows)
✅ Saved: data/raw/platform_claims.csv


In [6]:
# THE HEADLINE FINDING — promised vs reality
summary = df.groupby('platform').agg(
    daily_median_net=('net_earnings','median')
).reset_index()

summary['monthly_real'] = (summary['daily_median_net'] * 26).round(0)
summary['claimed']      = summary['platform'].map({
    p: d['claimed'] for p, d in platforms.items()
})
summary['gap_score']    = ((summary['claimed'] - summary['monthly_real']) / summary['claimed'] * 100).round(1)
summary['gap_rupees']   = (summary['claimed'] - summary['monthly_real']).round(0)

summary = summary.sort_values('gap_score', ascending=False)

print("=" * 60)
print("PROMISED VS REALITY — YOUR HEADLINE FINDING")
print("=" * 60)
for _, row in summary.iterrows():
    print(f"\n{row['platform']}")
    print(f"  Claimed:  ₹{row['claimed']:,.0f}/month")
    print(f"  Reality:  ₹{row['monthly_real']:,.0f}/month")
    print(f"  Gap:      {row['gap_score']}% (₹{row['gap_rupees']:,.0f} overstated)")

PROMISED VS REALITY — YOUR HEADLINE FINDING

Rapido
  Claimed:  ₹50,000/month
  Reality:  ₹15,705/month
  Gap:      68.6% (₹34,295 overstated)

Zomato
  Claimed:  ₹35,000/month
  Reality:  ₹13,153/month
  Gap:      62.4% (₹21,847 overstated)

Swiggy
  Claimed:  ₹40,000/month
  Reality:  ₹15,381/month
  Gap:      61.5% (₹24,619 overstated)

Porter
  Claimed:  ₹38,000/month
  Reality:  ₹18,439/month
  Gap:      51.5% (₹19,561 overstated)

Urban Company
  Claimed:  ₹45,000/month
  Reality:  ₹25,941/month
  Gap:      42.4% (₹19,059 overstated)
